# Scenario & Workload Generator

Use this notebook to generate scenario and workload matrices based on template YAML files.

In [2]:
# Install dependencies (PyYAML and ipywidgets for convenience)
%pip install pyyaml ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
from IPython.display import display, Markdown
import ipywidgets as widgets
import subprocess
import shlex

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCENARIO_TEMPLATE = PROJECT_ROOT / 'data' / 'scenarios' / 'scenario_2.yml'
WORKLOAD_TEMPLATE = PROJECT_ROOT / 'data' / 'workloads' / 'workload_1.yml'
SCENARIO_SCRIPT = PROJECT_ROOT / 'python' / 'generate_scenarios.py'
WORKLOAD_SCRIPT = PROJECT_ROOT / 'python' / 'generate_workloads.py'
PYTHON_EXECUTABLE = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'

## Scenario generation

In [9]:
scenario_template = widgets.Text(value=str(SCENARIO_TEMPLATE), description="Template")
scenario_output = widgets.Text(value=str(PROJECT_ROOT / 'data' / 'scenarios' / 'generated'), description="Output dir")
scenario_base_name = widgets.Text(value="scenario", description="Base name")
scenario_sets = widgets.Textarea(value="system.cores=2,4\ntiming.context_switch_cost_us=25,50", description="--set entries", layout=widgets.Layout(width='600px', height='120px'))
scenario_run = widgets.Button(description="Generate Scenarios", button_style="success")
scenario_log = widgets.Output()

def run_scenario(_):
    scenario_log.clear_output()
    with scenario_log:
        cmd = [
            str(PYTHON_EXECUTABLE),
            str(SCENARIO_SCRIPT),
            "--template", scenario_template.value,
            "--output-dir", scenario_output.value,
            "--base-name", scenario_base_name.value,
        ]
        for line in scenario_sets.value.splitlines():
            line = line.strip()
            if not line:
                continue
            cmd.extend(["--set", line])
        print("Running:", " ".join(shlex.quote(part) for part in cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            print(f"Generation failed: {exc}")

scenario_run.on_click(run_scenario)
display(widgets.VBox([
    scenario_template,
    scenario_output,
    scenario_base_name,
    scenario_sets,
    scenario_run,
    scenario_log
]))

## Workload generation

In [ ]:
workload_template = widgets.Text(value=str(WORKLOAD_TEMPLATE), description="Template")
workload_output = widgets.Text(value=str(PROJECT_ROOT / 'data' / 'workloads' / 'generated'), description="Output dir")
workload_base_name = widgets.Text(value="workload", description="Base name")
workload_sets = widgets.Textarea(value="tasks.0.exec_ms.min=2,3\ntasks.0.exec_ms.max=4,5\ntasks.0.priority=1,2", description="--set entries", layout=widgets.Layout(width='600px', height='140px'))
workload_run = widgets.Button(description="Generate Workloads", button_style="success")
workload_log = widgets.Output()

def run_workload(_):
    workload_log.clear_output()
    with workload_log:
        cmd = [
            str(PYTHON_EXECUTABLE),
            str(WORKLOAD_SCRIPT),
            "--template", workload_template.value,
            "--output-dir", workload_output.value,
            "--base-name", workload_base_name.value,
        ]
        for line in workload_sets.value.splitlines():
            line = line.strip()
            if not line:
                continue
            cmd.extend(["--set", line])
        print("Running:", " ".join(shlex.quote(part) for part in cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            print(f"Generation failed: {exc}")

workload_run.on_click(run_workload)
display(widgets.VBox([
    workload_template,
    workload_output,
    workload_base_name,
    workload_sets,
    workload_run,
    workload_log
]))